# Fine-tuning no Google Colab

Notebook-base para executar o treino a partir de `resources/finetuning_qa.jsonl` enviado manualmente ao Google Drive.

Fluxo:
1. Instalar dependências
2. Montar o Google Drive
3. Carregar o dataset JSONL
4. Configurar o modelo base
5. Executar o fine-tuning leve com QLoRA
6. Testar prompts após o treino


In [ ]:
!pip -q install --upgrade transformers datasets accelerate peft trl bitsandbytes sentencepiece huggingface_hub


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
from getpass import getpass
from pathlib import Path

from huggingface_hub import login

HF_TOKEN = os.getenv('HF_TOKEN') or getpass('Hugging Face token: ')
login(token=HF_TOKEN)

DATASET_PATH = Path('/content/drive/MyDrive/teach-chalenge3/finetuning_qa.jsonl')
OUTPUT_DIR = Path('/content/drive/MyDrive/teach-chalenge3/medqa-finetuned-model')
BASE_MODEL = 'meta-llama/Llama-3.2-1B-Instruct'
MAX_LENGTH = 512

assert DATASET_PATH.exists(), f'Dataset não encontrado em {DATASET_PATH}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(DATASET_PATH)
print(OUTPUT_DIR)


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(tokenizer.__class__.__name__)


In [ ]:
from datasets import load_dataset

dataset = load_dataset('json', data_files=str(DATASET_PATH), split='train')
dataset = dataset.train_test_split(test_size=0.05, seed=42)
dataset


In [ ]:
def preprocess_function(examples):
    tokenized = tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )
    tokenized['length'] = [len(ids) for ids in tokenized['input_ids']]
    return tokenized

tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=['source'])
tokenized_dataset = tokenized_dataset.filter(lambda example: example['length'] > 0)
tokenized_dataset


In [ ]:
from transformers import BitsAndBytesConfig, AutoModelForCausalLM

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype='bfloat16',
    bnb_4bit_quant_type='nf4',
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
)


In [ ]:
from peft import LoraConfig
from trl import SFTTrainer
from transformers import TrainingArguments

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=25,
    save_strategy='epoch',
    eval_strategy='epoch',
    bf16=True,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    peft_config=lora_config,
    processing_class=tokenizer,
)


In [ ]:
trainer.train()
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))


In [ ]:
from transformers import pipeline

generator = pipeline('text-generation', model=model, tokenizer=tokenizer, device_map='auto')
prompt = 'ANSWER THE QUESTION.\n[|Question|] What is the role of antibiotics in bacterial infections?[|eQuestion|]\n\n[|Answer|]'
print(generator(prompt, max_new_tokens=128, do_sample=False)[0]['generated_text'])
